# **Chapter 3: Adversarial Attacks and Model Security Lab** 🎯🛡️

---

## **Lab Overview**

Welcome to the Adversarial Attacks lab! This hands-on exercise will teach you how attackers can craft **imperceptible perturbations** that fool machine learning models into making incorrect predictions—and how to defend against these attacks.

### **What You'll Learn**
✅ How adversarial examples exploit gradient information to fool neural networks  
✅ Implementing **FGSM** (Fast Gradient Sign Method) and **PGD** (Projected Gradient Descent) attacks  
✅ Understanding **attack transferability** (black-box attacks across different models)  
✅ Building defenses: **adversarial training** and **input transformations**  
✅ Evaluating the robustness-accuracy tradeoff in secure AI systems

### **Security Context**
Adversarial attacks are a critical threat to deployed AI systems:
- **Autonomous vehicles**: Physical patches on stop signs causing misclassification
- **Face recognition**: Adversarial glasses/makeup to evade detection
- **Malware detection**: Perturbations to bypass antivirus ML models
- **Medical imaging**: Manipulated scans leading to misdiagnosis

### **Prerequisites**
- Python basics (variables, loops, functions)
- Basic understanding of neural networks (from Chapter 1)
- Familiarity with NumPy arrays and image data

### **Lab Structure**
1. Interactive demo exploration (visualize attacks in browser)
2. Build baseline neural network classifier
3. Implement gradient-based attacks (FGSM, PGD)
4. Test attack transferability (black-box scenario)
5. Deploy defenses (adversarial training, input transformations)
6. Compare attack/defense effectiveness

**Estimated Time:** 60-75 minutes

---

## **Interactive Demo: Visit adversarial.js Web Page** 🌐

Before diving into code, explore the interactive adversarial attack demonstration:

**URL:** https://kennysong.github.io/adversarial.js/

### **What to Try:**

1. Select a digit (0-9)
2. **Click "Generate Adversarial"** to create an adversarial perturbation
3. **Observe**:
   - The original prediction and confidence
   - The adversarial perturbation (amplified for visibility)
   - The adversarial example (original + perturbation)
   - How the prediction changes with minimal visible change
4. **Experiment** with different digits and attack strengths
5. **Notice** how the perturbation is imperceptible but causes misclassification
6. Try with the road signs

### **Key Insights to Observe:**
- Perturbations are often **invisible to human eyes** (< 1% pixel value change)
- Model confidence can be **very high on wrong predictions** (>95%)
- Different digits have different robustness levels
- Small epsilon values (0.1-0.3) are often sufficient

**Take 5-10 minutes to explore this demo before proceeding.**

---

Now let's build these attacks ourselves!

---

## **Step 1: Environment Setup & Data Loading** 📦

### **What We're Doing**
Setting up the Python environment and loading MNIST dataset. We'll use a **neural network** (not just logistic regression) because adversarial attacks require computing **gradients**, which neural networks provide.

### **Why Neural Networks?**
- Adversarial attacks exploit the model's decision boundary
- Gradient-based attacks need differentiable models
- Neural networks have complex, high-dimensional decision boundaries that are easier to fool

### **What to Expect**
- Import necessary libraries
- Load and normalize MNIST data
- Split into training/test sets

In [ ]:
# Import required libraries for ML, visualization, and attacks
import numpy as np                          # Numerical operations
import pandas as pd                         # Data manipulation
import matplotlib.pyplot as plt             # Visualization
import seaborn as sns                       # Statistical plots
from sklearn.datasets import fetch_openml  # MNIST dataset
from sklearn.model_selection import train_test_split  # Data splitting
from sklearn.metrics import accuracy_score, classification_report  # Evaluation
from sklearn.neural_network import MLPClassifier  # Neural network
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)
print("✓ All imports successful!")
print("✓ NumPy, Pandas, Matplotlib, Scikit-learn ready")
print("=" * 50)

# Set visualization style and random seed for reproducibility
sns.set(style="whitegrid")
np.random.seed(42)

In [ ]:
# Load MNIST dataset (same subset from Chapter 2 for consistency)
print("Loading MNIST dataset...")
print("(This may take 10-20 seconds on first run...)")

mnist = fetch_openml('mnist_784', version=1, parser='auto')
# Normalize pixel values to [0, 1] range (neural networks train better with normalized inputs)
X_full = mnist.data.astype('float32') / 255.0
y_full = mnist.target.astype('int')

# Use subset for faster training (10,000 samples)
# Full MNIST has 70,000 samples - we use smaller set for lab speed
np.random.seed(42)
subset_idx = np.random.choice(len(X_full), size=10000, replace=False)
X = X_full.iloc[subset_idx].reset_index(drop=True)
y = y_full.iloc[subset_idx].reset_index(drop=True)

# Train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% for testing
    random_state=42,    # Reproducibility
    stratify=y          # Keep class balance
)

print("\n" + "=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Total samples:     {len(X):,}")
print(f"Training samples:  {len(X_train):,} (80%)")
print(f"Test samples:      {len(X_test):,} (20%)")
print(f"Image dimensions:  28×28 pixels (784 features)")
print(f"Classes:           {sorted(y.unique())} (digits 0-9)")
print("=" * 50)
print("\n✓ Dataset loaded and split successfully!")

---

## **Step 2: Train a Neural Network Classifier** 🧠

### **What We're Doing**
Building a baseline neural network (Multi-Layer Perceptron) to classify handwritten digits. This model will be our **target** for adversarial attacks.

### **Why This Matters**
- We need gradients for adversarial attacks → neural networks provide this
- Baseline accuracy establishes the "clean" performance benchmark
- Real-world adversarial attacks target deployed neural networks (face recognition, malware detection, autonomous vehicles)

### **Architecture Details**
- **Input layer**: 784 neurons (28×28 pixel values)
- **Hidden layer 1**: 128 neurons with ReLU activation
- **Hidden layer 2**: 64 neurons with ReLU activation  
- **Output layer**: 10 neurons (one per digit class)

### **What to Expect**
- ~92-95% accuracy on clean test data
- High confidence predictions (often >90% probability)

In [ ]:
# Train a multi-layer perceptron (neural network)
print("Training neural network classifier...")
print("Architecture: 784 → 128 → 64 → 10")
print("(This may take 30-60 seconds...)")

model = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # Two hidden layers
    activation='relu',              # ReLU activation (standard for modern NNs)
    max_iter=20,                    # 20 training epochs
    random_state=42,                # Reproducibility
    verbose=False                   # Suppress training logs
)

model.fit(X_train, y_train)
print("✓ Training complete!")

# Evaluate baseline performance on clean test data
y_pred = model.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred)

print("\n" + "=" * 50)
print("BASELINE PERFORMANCE (Clean Data)")
print("=" * 50)
print(f"Test Accuracy: {baseline_acc:.4f} ({baseline_acc*100:.1f}%)")
print("=" * 50)

# Show sample predictions to understand model behavior
print("\nSample predictions (first 10 test samples):")
print(f"{'Sample':<8} {'True':<6} {'Pred':<6} {'Confidence':<12}")
print("-" * 40)

pred_probs = model.predict_proba(X_test[:10])
for i in range(10):
    true_label = y_test.iloc[i]
    pred_label = y_pred[i]
    confidence = pred_probs[i].max()
    status = "✓" if true_label == pred_label else "✗"
    print(f"{i:<8} {true_label:<6} {pred_label:<6} {confidence:<12.3f} {status}")

print("\n✓ Baseline model established. This is our attack target!")

---

## **Step 3: Implement FGSM (Fast Gradient Sign Method)** ⚡

### **What We're Doing**
Building our first adversarial attack! FGSM adds a small perturbation in the direction of the gradient to **maximize the model's loss** (confuse it).

### **The Math** (Simplified)
```
Adversarial Example = Original Image + ε × sign(Gradient of Loss)
```

- **Gradient**: Direction that increases the model's prediction error
- **sign()**: Just the direction (+1 or -1), not magnitude
- **ε (epsilon)**: Controls perturbation strength (e.g., 0.1 = 10% max change)

### **Why This Works**
Neural networks are **linear** in high-dimensional space. Moving slightly in the gradient direction pushes the input across the decision boundary, causing misclassification.

### **Security Implication**
FGSM is **fast** (single gradient computation) but effective. Attackers can generate adversarial examples in real-time against deployed models.

### **What We'll Measure**
- Attack Success Rate (ASR): % of samples where prediction changes
- Perturbation magnitude (L2 and L∞ norms)
- Accuracy drop on adversarial examples

In [ ]:
# Implement FGSM attack using numerical gradient approximation
# Note: We use finite differences since sklearn doesn't provide automatic differentiation

def compute_gradient_approx(model, x, y_true, epsilon=1e-5):
    """
    Approximate gradient of loss with respect to input using finite differences.
    
    Args:
        model: Trained classifier
        x: Input image (784-dim vector)
        y_true: True class label
        epsilon: Small step for numerical approximation
    
    Returns:
        Gradient vector (784-dim) showing loss sensitivity to each pixel
    """
    x_array = x.values if hasattr(x, 'values') else x
    gradient = np.zeros_like(x_array, dtype=float)
    
    # Get base loss (cross-entropy approximation using log probabilities)
    probs = model.predict_proba([x_array])[0]
    base_loss = -np.log(probs[y_true] + 1e-10)  # Avoid log(0)
    
    # Compute gradient for each pixel (finite difference method)
    for i in range(len(x_array)):
        x_perturbed = x_array.copy()
        x_perturbed[i] += epsilon  # Perturb single pixel
        
        probs_perturbed = model.predict_proba([x_perturbed])[0]
        perturbed_loss = -np.log(probs_perturbed[y_true] + 1e-10)
        
        # Gradient = change in loss / change in pixel value
        gradient[i] = (perturbed_loss - base_loss) / epsilon
    
    return gradient

def fgsm_attack(model, x, y_true, epsilon=0.1):
    """
    Generate FGSM adversarial example.
    
    Args:
        model: Target classifier to attack
        x: Original input image
        y_true: True class label
        epsilon: Perturbation strength (0.1 = 10% max change)
    
    Returns:
        x_adv: Adversarial example
        perturbation: Added noise (for visualization)
    """
    gradient = compute_gradient_approx(model, x, y_true)
    # FGSM formula: sign of gradient scaled by epsilon
    perturbation = epsilon * np.sign(gradient)
    x_array = x.values if hasattr(x, 'values') else x
    # Add perturbation and clip to valid pixel range [0, 1]
    x_adv = np.clip(x_array + perturbation, 0, 1)
    return x_adv, perturbation

print("=" * 50)
print("FGSM ATTACK IMPLEMENTATION")
print("=" * 50)
print("✓ Gradient approximation function ready")
print("✓ FGSM attack function ready")
print("=" * 50)
print("\nℹ️  Note: This uses numerical gradient approximation (finite differences).")
print("   In production, use TensorFlow/PyTorch with automatic differentiation for speed.")

In [ ]:
# Generate FGSM adversarial examples for test set
print("Generating FGSM adversarial examples...")
print("Processing first 100 test samples...")
print("(This may take 1-2 minutes due to gradient computation...)\n")

epsilon = 0.15  # Perturbation strength
n_samples = 100
adversarial_examples = []
perturbations = []

for i in range(n_samples):
    # Progress indicator every 20 samples
    if (i + 1) % 20 == 0:
        print(f"  Progress: {i+1}/{n_samples} ({(i+1)/n_samples*100:.0f}%)")
    
    x_orig = X_test.iloc[i]
    y_true = y_test.iloc[i]
    
    # Generate adversarial example using FGSM
    x_adv, pert = fgsm_attack(model, x_orig, y_true, epsilon)
    adversarial_examples.append(x_adv)
    perturbations.append(pert)

adversarial_examples = np.array(adversarial_examples)
perturbations = np.array(perturbations)

print(f"\n" + "=" * 50)
print(f"✓ Generated {len(adversarial_examples)} adversarial examples")
print(f"✓ Perturbation strength (epsilon): {epsilon}")
print("=" * 50)

In [ ]:
# Evaluate FGSM attack success
print("Evaluating FGSM attack effectiveness...\n")

# Get predictions on original vs adversarial examples
y_pred_adv = model.predict(adversarial_examples)
y_pred_orig = model.predict(X_test.iloc[:n_samples])
y_true_subset = y_test.iloc[:n_samples]

# Calculate attack metrics
orig_acc = accuracy_score(y_true_subset, y_pred_orig)
adv_acc = accuracy_score(y_true_subset, y_pred_adv)
attack_success_rate = (y_pred_orig != y_pred_adv).mean()

print("=" * 50)
print(f"FGSM ATTACK RESULTS (ε={epsilon})")
print("=" * 50)
print(f"Original Accuracy:      {orig_acc:.4f} ({orig_acc*100:.1f}%)")
print(f"Adversarial Accuracy:   {adv_acc:.4f} ({adv_acc*100:.1f}%)")
print(f"Accuracy Drop:          {orig_acc - adv_acc:.4f} ({(orig_acc - adv_acc)*100:.1f}%)")
print(f"\n⚠️  Attack Success Rate: {attack_success_rate:.4f} ({attack_success_rate*100:.1f}%)")
print(f"    ({int(attack_success_rate*n_samples)}/{n_samples} predictions changed)")
print("=" * 50)

# Perturbation statistics (measure how "visible" the attack is)
l2_norms = [np.linalg.norm(p) for p in perturbations]
linf_norms = [np.max(np.abs(p)) for p in perturbations]

print(f"\nPerturbation Statistics:")
print(f"  Mean L2 norm:    {np.mean(l2_norms):.4f}")
print(f"  Mean L∞ norm:    {np.mean(linf_norms):.4f}")
print(f"\nℹ️  Interpretation:")
print(f"  - Low L∞ norm ({np.mean(linf_norms):.3f}) means perturbations are imperceptible")
print(f"  - High ASR ({attack_success_rate*100:.0f}%) means attack is effective")
print(f"  - This is why adversarial attacks are dangerous!")

---

## **Step 4: Visualize Adversarial Examples** 👁️

### **What We're Doing**
Visualizing successful adversarial attacks to see what imperceptible perturbations look like.

### **Why This Matters**
- **Human perception vs model perception**: We can't see the attack, but the model is fooled
- **Security implication**: Attackers can manipulate images without raising human suspicion
- **Real-world risk**: Physical adversarial patches on road signs, clothing, etc.

### **What to Look For**
- Original and adversarial images should look **nearly identical**
- Perturbation (when amplified 10×) shows which pixels were changed
- Model predictions flip despite minimal visual change

In [ ]:
# Visualize successful adversarial examples
successful_attacks = np.where(y_pred_orig != y_pred_adv)[0]
n_viz = min(6, len(successful_attacks))

print(f"Visualizing {n_viz} successful attacks (where prediction changed)...\n")

fig, axes = plt.subplots(n_viz, 4, figsize=(12, 3*n_viz))
if n_viz == 1:
    axes = axes.reshape(1, -1)

for i, idx in enumerate(successful_attacks[:n_viz]):
    # Column 1: Original image
    orig = X_test.iloc[idx].values.reshape(28, 28)
    axes[i, 0].imshow(orig, cmap='gray')
    axes[i, 0].set_title(f"Original\nPred: {y_pred_orig[idx]} | True: {y_true_subset.iloc[idx]}", 
                         fontweight='bold')
    axes[i, 0].axis('off')
    
    # Column 2: Perturbation (amplified 10× for visibility)
    pert = perturbations[idx].reshape(28, 28)
    axes[i, 1].imshow(pert * 10, cmap='seismic', vmin=-1, vmax=1)
    axes[i, 1].set_title("Perturbation\n(10× amplified)", fontweight='bold')
    axes[i, 1].axis('off')
    
    # Column 3: Adversarial image
    adv = adversarial_examples[idx].reshape(28, 28)
    axes[i, 2].imshow(adv, cmap='gray')
    axes[i, 2].set_title(f"Adversarial\nPred: {y_pred_adv[idx]} ⚠️", 
                         fontweight='bold', color='red')
    axes[i, 2].axis('off')
    
    # Column 4: Difference (amplified 10×)
    diff = (adv - orig)
    axes[i, 3].imshow(diff * 10, cmap='seismic', vmin=-1, vmax=1)
    axes[i, 3].set_title("Difference\n(10× amplified)", fontweight='bold')
    axes[i, 3].axis('off')

plt.suptitle(f"FGSM Adversarial Examples (ε={epsilon})", fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"\n✓ Visualization complete!")
print(f"\nℹ️  Key Observations:")
print(f"  1. Original and adversarial images look nearly IDENTICAL to human eyes")
print(f"  2. Perturbation is IMPERCEPTIBLE (needs 10× amplification to see)")
print(f"  3. Model predictions FLIP despite minimal visual change")
print(f"  4. This demonstrates the vulnerability of neural networks to adversarial attacks!")

---

## **Step 5: Iterative Attack - Projected Gradient Descent (PGD)** 🔄🎯

### **What We're Doing**
Implementing a **stronger** attack that applies FGSM iteratively. PGD refines the adversarial example over multiple steps, making it more effective than single-step FGSM.

### **How PGD Works**
1. Start with original image
2. Take small step in gradient direction (like FGSM, but smaller)
3. **Project** back to epsilon ball (ensure total perturbation ≤ ε)
4. Repeat for multiple iterations
5. Result: More effective adversarial example within same perturbation budget

### **The Math**
```
For i = 1 to num_iterations:
    x = x + α × sign(∇Loss(x))        # Take small step
    x = clip(x, x_orig - ε, x_orig + ε)  # Project to epsilon ball
    x = clip(x, 0, 1)                  # Ensure valid pixel range
```

### **Why PGD is Stronger**
- **Multiple iterations** escape local minima
- **Projection** ensures optimal use of perturbation budget
- **Considered the strongest first-order attack** (commonly used for robustness evaluation)

### **Security Implication**
PGD is the **gold standard** for evaluating adversarial robustness. If your model is robust to PGD, it's likely robust to other gradient-based attacks.

In [ ]:
# Implement PGD (Projected Gradient Descent) attack
def pgd_attack(model, x, y_true, epsilon=0.15, alpha=0.01, num_iter=10):
    """
    Projected Gradient Descent attack - iterative FGSM with projection.
    
    Args:
        model: Target classifier
        x: Original input image
        y_true: True class label
        epsilon: Maximum total perturbation (L∞ bound)
        alpha: Step size per iteration (typically epsilon/num_iter)
        num_iter: Number of iterations
    
    Returns:
        x_adv: Adversarial example
        perturbation: Total perturbation added
    """
    x_array = x.values if hasattr(x, 'values') else x
    x_adv = x_array.copy()
    
    for iteration in range(num_iter):
        # Step 1: Compute gradient at current point
        gradient = compute_gradient_approx(model, x_adv, y_true, epsilon=1e-5)
        
        # Step 2: Take step in gradient direction (like FGSM, but smaller)
        x_adv = x_adv + alpha * np.sign(gradient)
        
        # Step 3: Project back to epsilon ball around original image
        # (ensure total perturbation doesn't exceed epsilon)
        perturbation = np.clip(x_adv - x_array, -epsilon, epsilon)
        x_adv = x_array + perturbation
        
        # Step 4: Clip to valid pixel range [0, 1]
        x_adv = np.clip(x_adv, 0, 1)
    
    final_perturbation = x_adv - x_array
    return x_adv, final_perturbation

print("=" * 50)
print("PGD ATTACK IMPLEMENTATION")
print("=" * 50)
print("✓ PGD attack function ready")
print("=" * 50)

print(f"\nℹ️  Why PGD is stronger than FGSM:")
print(f"  1. Multiple iterations refine the adversarial example")
print(f"  2. Projection ensures optimal use of perturbation budget")
print(f"  3. Can escape local minima to find better attacks")
print(f"  4. Gold standard for robustness evaluation in research")

In [ ]:
# Generate PGD adversarial examples (smaller set due to computation time)
print("Generating PGD adversarial examples...")
print("Processing first 50 test samples (10 iterations each)...")
print("(This may take 2-3 minutes...)\n")

n_pgd = 50
pgd_examples = []
pgd_perturbations = []

for i in range(n_pgd):
    if (i + 1) % 10 == 0:
        print(f"  Progress: {i+1}/{n_pgd} ({(i+1)/n_pgd*100:.0f}%)")
    
    x_orig = X_test.iloc[i]
    y_true = y_test.iloc[i]
    
    # Generate PGD adversarial example
    # alpha = epsilon/num_iter ensures we don't overshoot
    x_pgd, pert_pgd = pgd_attack(model, x_orig, y_true, epsilon=0.15, alpha=0.02, num_iter=10)
    pgd_examples.append(x_pgd)
    pgd_perturbations.append(pert_pgd)

pgd_examples = np.array(pgd_examples)
pgd_perturbations = np.array(pgd_perturbations)

print(f"\n" + "=" * 50)
print(f"✓ Generated {len(pgd_examples)} PGD adversarial examples")
print("=" * 50)

# Evaluate PGD attack effectiveness
y_pred_pgd = model.predict(pgd_examples)
y_pred_orig_pgd = model.predict(X_test.iloc[:n_pgd])
y_true_pgd = y_test.iloc[:n_pgd]

orig_acc_pgd = accuracy_score(y_true_pgd, y_pred_orig_pgd)
pgd_acc = accuracy_score(y_true_pgd, y_pred_pgd)
pgd_success = (y_pred_orig_pgd != y_pred_pgd).mean()

print(f"\nPGD Attack Results:")
print(f"  Original Accuracy:      {orig_acc_pgd:.4f} ({orig_acc_pgd*100:.1f}%)")
print(f"  PGD Adversarial Acc:    {pgd_acc:.4f} ({pgd_acc*100:.1f}%)")
print(f"  Attack Success Rate:    {pgd_success:.4f} ({pgd_success*100:.1f}%)")

print(f"\n" + "=" * 50)
print("FGSM vs PGD COMPARISON")
print("=" * 50)
print(f"  FGSM Success Rate: {attack_success_rate:.4f} ({attack_success_rate*100:.1f}%)")
print(f"  PGD Success Rate:  {pgd_success:.4f} ({pgd_success*100:.1f}%)")

if pgd_success > attack_success_rate:
    improvement = (pgd_success - attack_success_rate) * 100
    print(f"\n✓ PGD is {improvement:.1f}% more effective than FGSM!")
else:
    print(f"\nℹ️  Both attacks are similarly effective on this model")
print("=" * 50)

---

## **Step 6: Adversarial Transferability (Black-Box Attack)** 🔓

### **What We're Doing**
Testing if adversarial examples generated for **Model 1** also fool **Model 2** (different architecture). This simulates a **black-box attack** where the attacker doesn't know the target model's internals.

### **Why This Matters for Security**
- **Real-world scenarios**: Attacker generates adversarials on their own model (white-box), then attacks deployed model (black-box)
- **Defense evasion**: Even if you keep model architecture secret, transferability allows attacks
- **Example**: Adversarial examples for one face recognition system may fool competitors' systems

### **What We'll Test**
- Train second model with different architecture (256→128 vs 128→64)
- Test FGSM examples (generated for Model 1) on Model 2
- Measure transfer success rate

### **Expected Outcome**
Adversarial examples will **partially transfer**—not 100% effective, but significantly better than random chance. This proves black-box attacks are viable.

In [ ]:
# Train a second model with DIFFERENT architecture (to test transferability)
print("Training second model with different architecture...")
print("Model 1: 784 → 128 → 64 → 10")
print("Model 2: 784 → 256 → 128 → 10  (wider layers)")
print("(This simulates a different vendor's model...)\n")

model2 = MLPClassifier(
    hidden_layer_sizes=(256, 128),  # Different from Model 1!
    activation='relu',
    max_iter=20,
    random_state=123,  # Different random seed
    verbose=False
)

model2.fit(X_train, y_train)
print("✓ Model 2 training complete!")

# Evaluate Model 2 on clean data
y_pred_model2 = model2.predict(X_test)
model2_acc = accuracy_score(y_test, y_pred_model2)

print("\n" + "=" * 50)
print("MODEL 2 BASELINE PERFORMANCE")
print("=" * 50)
print(f"Clean Test Accuracy: {model2_acc:.4f} ({model2_acc*100:.1f}%)")
print("=" * 50)
print("\n✓ Model 2 ready for transferability test!")

In [ ]:
# Test transferability: Do FGSM examples (from Model 1) fool Model 2?
print("Testing transferability of adversarial examples...\n")

# Model 2 predictions on adversarial examples (generated for Model 1)
y_pred_model2_adv = model2.predict(adversarial_examples)
y_pred_model2_orig = model2.predict(X_test.iloc[:n_samples])

# Calculate transfer success rate
transfer_success = (y_pred_model2_orig != y_pred_model2_adv).mean()
model2_adv_acc = accuracy_score(y_true_subset, y_pred_model2_adv)

print("=" * 60)
print("TRANSFERABILITY TEST RESULTS")
print("=" * 60)

print(f"\nModel 1 (SOURCE - used to generate adversarial examples):")
print(f"  Clean accuracy:       {orig_acc:.4f}")
print(f"  Adversarial accuracy: {adv_acc:.4f}")
print(f"  Attack success rate:  {attack_success_rate:.4f} ({attack_success_rate*100:.0f}%)")

print(f"\nModel 2 (TARGET - different architecture, never seen adversarials):")
print(f"  Clean accuracy:       {accuracy_score(y_true_subset, y_pred_model2_orig):.4f}")
print(f"  Adversarial accuracy: {model2_adv_acc:.4f}")
print(f"  Transfer success:     {transfer_success:.4f} ({transfer_success*100:.0f}%)")

# Calculate transfer rate (percentage of successful attacks that transfer)
if attack_success_rate > 0:
    transfer_rate = transfer_success / attack_success_rate
    print(f"\n⚠️  Transferability Rate: {transfer_rate*100:.0f}%")
    print(f"    ({int(transfer_success*n_samples)}/{int(attack_success_rate*n_samples)} attacks that fooled Model 1 also fool Model 2)")
else:
    print("\nNo successful attacks to transfer.")

print("=" * 60)

print(f"\nℹ️  Key Insight:")
print(f"  Adversarial examples TRANSFER between models with different architectures!")
print(f"  This enables BLACK-BOX attacks without knowing target model details.")
print(f"  Security implication: Keeping model architecture secret is NOT sufficient defense.")

---

## **Step 7: Defense - Adversarial Training** 🛡️💪

### **What We're Doing**
Training a model on a **mixture of clean AND adversarial examples** to improve robustness. This is the most effective defense against adversarial attacks.

### **How It Works**
1. Generate adversarial examples from training data
2. Combine clean + adversarial samples
3. Train model on this augmented dataset
4. Model learns to be robust to perturbations

### **Why This Works**
- Model sees adversarial perturbations during training
- Learns decision boundaries that are more stable
- Similar to data augmentation, but adversarially-aware

### **The Tradeoff**
- ✅ **Pro**: Significantly improves adversarial robustness
- ⚠️ **Con**: May slightly reduce clean accuracy
- ⚠️ **Con**: Increases training time (need to generate adversarials)

### **Real-World Use**
- Critical systems (autonomous vehicles, medical imaging)
- High-security applications (face recognition, malware detection)
- When adversarial robustness is prioritized over peak accuracy

In [ ]:
# Generate adversarial training data (smaller set for lab speed)
print("Generating adversarial training examples...")
print("Creating adversarial examples for 500 random training samples...")
print("(This may take 2-3 minutes...)\n")

n_adv_train = 500
X_train_adv_samples = []
y_train_adv_samples = []

for i in range(n_adv_train):
    if (i + 1) % 100 == 0:
        print(f"  Progress: {i+1}/{n_adv_train} ({(i+1)/n_adv_train*100:.0f}%)")
    
    # Randomly select training sample
    idx = np.random.randint(len(X_train))
    x_orig = X_train.iloc[idx]
    y_true = y_train.iloc[idx]
    
    # Generate adversarial example using FGSM
    x_adv, _ = fgsm_attack(model, x_orig, y_true, epsilon=0.15)
    X_train_adv_samples.append(x_adv)
    y_train_adv_samples.append(y_true)  # Keep original label!

X_train_adv = np.array(X_train_adv_samples)
y_train_adv = np.array(y_train_adv_samples)

# Combine clean and adversarial training data
X_train_combined = np.vstack([X_train.values, X_train_adv])
y_train_combined = np.concatenate([y_train.values, y_train_adv])

print(f"\n" + "=" * 50)
print("ADVERSARIAL TRAINING DATASET")
print("=" * 50)
print(f"Clean samples:       {len(X_train):,}")
print(f"Adversarial samples: {len(X_train_adv):,}")
print(f"Total samples:       {len(X_train_combined):,}")
print("=" * 50)
print("\nℹ️  Note: Adversarial examples keep their ORIGINAL labels")
print("   (We're teaching the model to classify perturbed inputs correctly)")

In [ ]:
# Train robust model with adversarial training
print("Training robust model with adversarial training...")
print("(Training on combined clean + adversarial dataset...)\n")

robust_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    max_iter=20,
    random_state=42,
    verbose=False
)

robust_model.fit(X_train_combined, y_train_combined)
print("✓ Adversarial training complete!")

# Evaluate robust model on CLEAN test data
y_pred_robust_clean = robust_model.predict(X_test.iloc[:n_samples])
robust_clean_acc = accuracy_score(y_true_subset, y_pred_robust_clean)

# Evaluate robust model on ADVERSARIAL test data
y_pred_robust_adv = robust_model.predict(adversarial_examples)
robust_adv_acc = accuracy_score(y_true_subset, y_pred_robust_adv)

print("\n" + "=" * 60)
print("DEFENSE EVALUATION: Adversarial Training")
print("=" * 60)

# Create comparison table
print(f"\n{'Model':<30} {'Clean Acc':<12} {'Adv Acc':<12} {'Robustness':<12}")
print("-" * 70)
print(f"{'Standard (no defense)':<30} {orig_acc:<12.4f} {adv_acc:<12.4f} {adv_acc:<12.4f}")
print(f"{'Adversarially Trained':<30} {robust_clean_acc:<12.4f} {robust_adv_acc:<12.4f} {robust_adv_acc:<12.4f}")
print("-" * 70)

# Calculate improvements
robustness_improvement = robust_adv_acc - adv_acc
clean_acc_change = robust_clean_acc - orig_acc

print(f"\nRobustness improvement: +{robustness_improvement:.4f} ({robustness_improvement/adv_acc*100:+.1f}% relative gain)")
print(f"Clean accuracy change:  {clean_acc_change:+.4f} ({clean_acc_change/orig_acc*100:+.1f}% relative change)")

print("=" * 60)

if robust_adv_acc > adv_acc * 1.1:
    print("\n✓ Adversarial training significantly improved robustness!")
else:
    print("\nℹ️  Moderate robustness improvement observed")

print("\nℹ️  Typical tradeoff:")
print("   - Adversarial accuracy improves (model more robust)")
print("   - Clean accuracy may decrease slightly (accepted cost for security)")

---

## **Step 8: Defense - Input Transformation** 🔧🛡️

### **What We're Doing**
Applying **preprocessing transformations** to adversarial examples to destroy perturbations before feeding to the model. Think of it as "cleaning" the input.

### **Transformation Techniques**

**1. Bit Depth Reduction**
- Reduce pixel precision (e.g., 8-bit → 4-bit)
- Rounds pixel values to coarser levels
- Destroys fine-grained adversarial perturbations

**2. Median Filtering**
- Replace each pixel with median of neighbors
- Smooths out localized noise
- Reduces high-frequency perturbations

### **Why This Works**
Adversarial perturbations are often **high-frequency** and **small-magnitude** changes. Transformations destroy these subtle patterns while preserving main image content.

### **The Tradeoff**
- ✅ **Pro**: Simple, no model retraining needed
- ✅ **Pro**: Can defend against unknown attacks
- ⚠️ **Con**: May hurt clean accuracy (legitimate inputs also transformed)
- ⚠️ **Con**: Attackers can adapt (generate robust adversarials that survive transformations)

### **Real-World Use**
- Preprocessing pipeline for deployed models
- Combined with other defenses (defense-in-depth)
- JPEG compression for image-based systems

In [ ]:
# Implement input transformation defenses

def bit_depth_reduction(x, bits=4):
    """
    Reduce bit depth to destroy fine-grained perturbations.
    
    Args:
        x: Input image (normalized 0-1)
        bits: Target bit depth (e.g., 4 = 16 levels instead of 256)
    
    Returns:
        Quantized image
    """
    levels = 2 ** bits
    # Round to nearest level, then scale back to [0, 1]
    x_quantized = np.round(x * (levels - 1)) / (levels - 1)
    return x_quantized

def median_filter_2d(x, size=3):
    """
    Apply median filter to smooth out perturbations.
    
    Args:
        x: Input image (flattened 784-dim)
        size: Filter window size (3 = 3×3 window)
    
    Returns:
        Filtered image (flattened)
    """
    from scipy.ndimage import median_filter
    img = x.reshape(28, 28)  # Reshape to 2D
    filtered = median_filter(img, size=size)
    return filtered.flatten()

print("=" * 50)
print("INPUT TRANSFORMATION DEFENSES")
print("=" * 50)
print("✓ Bit depth reduction function ready")
print("✓ Median filter function ready")
print("=" * 50)

# Apply transformations to adversarial examples
print("\nApplying transformations to adversarial examples...")

# Defense 1: Bit depth reduction (8-bit → 4-bit)
adv_quantized = np.array([bit_depth_reduction(x, bits=4) for x in adversarial_examples])
y_pred_quantized = model.predict(adv_quantized)
acc_quantized = accuracy_score(y_true_subset, y_pred_quantized)

print("✓ Bit depth reduction applied (8-bit → 4-bit)")

# Defense 2: Median filtering (if scipy available)
try:
    from scipy.ndimage import median_filter
    adv_filtered = np.array([median_filter_2d(x, size=3) for x in adversarial_examples])
    y_pred_filtered = model.predict(adv_filtered)
    acc_filtered = accuracy_score(y_true_subset, y_pred_filtered)
    has_scipy = True
    print("✓ Median filtering applied (3×3 window)")
except ImportError:
    print("⚠️  scipy not available, skipping median filter")
    has_scipy = False

print("\n" + "=" * 60)
print("INPUT TRANSFORMATION RESULTS")
print("=" * 60)
print(f"\n{'Defense Method':<30} {'Accuracy':<12} {'Improvement':<15}")
print("-" * 60)
print(f"{'No defense (adversarial)':<30} {adv_acc:<12.4f} {'baseline':<15}")
print(f"{'Bit depth reduction (4-bit)':<30} {acc_quantized:<12.4f} {f'+{acc_quantized-adv_acc:.4f}':<15}")
if has_scipy:
    print(f"{'Median filter (3×3)':<30} {acc_filtered:<12.4f} {f'+{acc_filtered-adv_acc:.4f}':<15}")
print("-" * 60)

print(f"\nℹ️  Key Insights:")
print(f"  - Simple transformations CAN reduce adversarial effectiveness")
print(f"  - Tradeoff: May also reduce clean accuracy slightly")
print(f"  - Best used in combination with other defenses (defense-in-depth)")

---

## **Step 9: Summary & Comparison** 📊

### **What We're Doing**
Visualizing all attack and defense results in a comprehensive comparison to understand the security landscape.

### **What This Shows**
- Attack effectiveness (how much accuracy drops)
- Defense effectiveness (how much robustness improves)
- Tradeoffs between different approaches

### **Interpretation Guide**
- **Higher is better** for all bars (all represent accuracy)
- **Baseline** (green) = clean data performance
- **Attack** bars (red) = model under attack
- **Defense** bars (blue) = defended model under attack

In [ ]:
# Create comprehensive summary of all experiments
print("Generating comprehensive results summary...\n")

results = pd.DataFrame([
    {'Scenario': 'Baseline (Clean)', 'Accuracy': baseline_acc, 'Category': 'Baseline'},
    {'Scenario': 'FGSM Attack', 'Accuracy': adv_acc, 'Category': 'Attack'},
    {'Scenario': 'PGD Attack', 'Accuracy': pgd_acc, 'Category': 'Attack'},
    {'Scenario': 'Transfer (Model 2)', 'Accuracy': model2_adv_acc, 'Category': 'Attack'},
    {'Scenario': 'Adversarial Training', 'Accuracy': robust_adv_acc, 'Category': 'Defense'},
    {'Scenario': 'Bit Depth Reduction', 'Accuracy': acc_quantized, 'Category': 'Defense'},
])

if has_scipy:
    results = pd.concat([results, pd.DataFrame([
        {'Scenario': 'Median Filter', 'Accuracy': acc_filtered, 'Category': 'Defense'}
    ])], ignore_index=True)

print("=" * 60)
print("COMPREHENSIVE RESULTS SUMMARY")
print("=" * 60)
display(results)
print("=" * 60)

# Visualize results
fig, ax = plt.subplots(figsize=(12, 6))
colors = {'Baseline': 'green', 'Attack': 'red', 'Defense': 'blue'}
bar_colors = [colors[cat] for cat in results['Category']]

bars = ax.bar(results['Scenario'], results['Accuracy'], color=bar_colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Accuracy', fontsize=13, fontweight='bold')
ax.set_xlabel('Scenario', fontsize=13, fontweight='bold')
ax.set_title('Adversarial Attack & Defense Comparison', fontsize=15, fontweight='bold', pad=20)
ax.axhline(baseline_acc, color='gray', linestyle='--', alpha=0.5, linewidth=2, label=f'Baseline ({baseline_acc:.3f})')
ax.set_ylim([0, 1.0])
ax.legend(fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add category legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', alpha=0.7, label='Baseline (clean data)'),
    Patch(facecolor='red', alpha=0.7, label='Attacks'),
    Patch(facecolor='blue', alpha=0.7, label='Defenses')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete!")

---

## **Step 10: Return to Interactive Demo** 🔄🌐

Now that you've implemented adversarial attacks yourself, revisit the web demo with deeper understanding:

**URL:** https://kennysong.github.io/adversarial.js/

---

### **New Insights to Explore:**

1. **Compare to your results**: The demo uses a similar approach (gradient-based perturbation)
   - Notice how epsilon controls visibility vs effectiveness (same tradeoff you saw!)
   - Try epsilon = 0.15 to match your experiments

2. **Observe transferability**: The demo's perturbations work because of shared decision boundaries
   - This is why your attacks transferred between Model 1 and Model 2
   - Real-world implication: Black-box attacks are viable!

3. **Try different epsilon values**: See how attack strength affects success rate and visibility
   - Small (0.05-0.1): Less visible, may fail on some samples
   - Medium (0.15-0.2): Good balance (your experiments used 0.15)
   - Large (0.3+): Very effective but more visible

4. **Think about defenses**: 
   - Would bit-depth reduction help? (Try mentally "rounding" pixel values)
   - Would median filtering help? (Imagine smoothing the image)
   - Could adversarial training make the model robust?

5. **Real-time visualization**:
   - Watch gradient flow in real-time
   - See how confidence changes with perturbation
   - Understand why linear models are vulnerable in high dimensions

---

### **Reflection Questions:**

**Attack Understanding:**
- How does the web demo's real-time visualization help understand gradient flow?
- Why do some digits (like 1, 7) seem easier to attack than others (like 0, 8)?
- What happens when you increase epsilon beyond 0.3? Is the attack still "imperceptible"?

**Security Implications:**
- What security implications does transferability have for deployed ML systems?
- If a company keeps its model architecture secret, are adversarial attacks still possible?
- How would you detect adversarial examples in production?

**Defense Strategies:**
- What trade-offs exist between model accuracy and adversarial robustness?
- Should self-driving cars prioritize clean accuracy or adversarial robustness? Why?
- Can you combine multiple defenses? What would that look like?

**Real-World Scenarios:**
- How would an attacker use adversarial examples against a face recognition system?
- Could adversarial patches on road signs fool autonomous vehicles? (Research "adversarial stop signs")
- How might malware authors use adversarial techniques to evade ML-based antivirus?

---

**Take 5-10 minutes to explore these questions with the interactive demo.**

---

## **🎯 Lab Summary: Key Takeaways**

### **What You Accomplished**

✅ Built baseline neural network with ~92-95% accuracy on clean MNIST data  
✅ Implemented **FGSM** (Fast Gradient Sign Method) adversarial attack  
✅ Implemented **PGD** (Projected Gradient Descent) - stronger iterative attack  
✅ Demonstrated **attack transferability** across different model architectures (black-box attacks)  
✅ Built **adversarial training** defense to improve robustness  
✅ Applied **input transformation** defenses (bit depth reduction, median filtering)  
✅ Compared attack/defense effectiveness across multiple scenarios

---

### **Security Lessons**

**1. Neural Networks Are Vulnerable to Adversarial Attacks**
- Imperceptible perturbations (ε = 0.15 ≈ 15% max pixel change) fool models
- High-confidence wrong predictions (model is "confidently wrong")
- Attacks are **fast** (FGSM takes milliseconds) and **effective** (50-80% success rate)

**2. Gradient-Based Attacks Exploit Model Decision Boundaries**
- Neural networks are locally linear in high-dimensional space
- Moving in gradient direction pushes inputs across decision boundaries
- More complex models (deeper networks) can be MORE vulnerable

**3. Transferability Enables Black-Box Attacks**
- Adversarial examples transfer between models with ~40-70% success rate
- Attacker can generate adversarials on surrogate model, attack production model
- **Security implication**: Keeping model architecture secret is NOT sufficient defense

**4. Defenses Exist But Have Tradeoffs**

**Adversarial Training:**
- ✅ Most effective defense (improves robustness significantly)
- ⚠️ Reduces clean accuracy slightly (2-5% typical)
- ⚠️ Increases training time (must generate adversarials)
- **Use when**: Security-critical applications (medical, autonomous vehicles)

**Input Transformations:**
- ✅ Simple, no retraining needed
- ✅ Can defend against unknown attacks
- ⚠️ May hurt clean accuracy
- ⚠️ Attackers can adapt (generate transformation-robust adversarials)
- **Use when**: Quick defense, preprocessing pipeline

**5. No Silver Bullet - Defense-in-Depth Required**
- No single defense is perfect
- Combine multiple techniques: adversarial training + input transformations + anomaly detection
- Monitor predictions in production (detect unusual confidence patterns)
- Regular adversarial testing (like penetration testing)

---

### **Real-World Attack Examples**

**Autonomous Vehicles:**
- Physical adversarial patches on stop signs → misclassified as speed limit
- Research: "Robust Physical-World Attacks on Deep Learning Visual Classification" (2018)
- Defense: Multi-sensor fusion, adversarial training on physical perturbations

**Face Recognition:**
- Adversarial glasses/makeup to evade detection
- Research: "Accessorize to a Crime" (Carnegie Mellon, 2016)
- Defense: Liveness detection, ensemble models, adversarial training

**Malware Detection:**
- Adversarial perturbations to binary files → evade ML antivirus
- Research: "Adversarial Examples for Malware Detection" (2017)
- Defense: Hybrid ML + signature-based detection, input validation

**Medical Imaging:**
- Adversarial perturbations to CT/MRI scans → misdiagnosis
- Research: "Adversarial Attacks Against Medical Deep Learning Systems" (2019)
- Defense: Physician-in-the-loop, adversarial training, certified defenses

**Spam/Phishing Filters:**
- Adversarial text modifications → evade detection
- Research: "Generating Natural Language Adversarial Examples" (2018)
- Defense: Ensemble models, human review for suspicious cases

---

### **Next Steps for Real-World Application**

**For Security Analysts:**
- **Threat modeling**: Include adversarial attacks in AI/ML system threat models
- **Monitoring**: Track prediction confidence distributions (detect anomalies)
- **Testing**: Conduct adversarial robustness testing before deployment
- **Incident response**: Plan for adversarial attack scenarios

**For ML Engineers:**
- **Adversarial training**: Incorporate into training pipeline for critical systems
- **Defense-in-depth**: Combine adversarial training + input transformations + ensembles
- **Evaluation**: Test robustness with PGD attacks (gold standard)
- **Certified defenses**: Explore provable robustness (randomized smoothing, interval bound propagation)

**For Organizations:**
- **Risk assessment**: Identify ML systems vulnerable to adversarial attacks
- **Security requirements**: Define adversarial robustness requirements for AI systems
- **Red teaming**: Conduct adversarial attack simulations
- **Policy**: Establish guidelines for adversarially robust AI development

---

### **Try on Your Own (Extensions)**

**Experiment with Parameters:**
- Vary epsilon (ε ∈ {0.05, 0.1, 0.2, 0.3}) and plot accuracy vs epsilon curve
- Try different PGD iterations (5, 10, 20, 50) and compare effectiveness
- Test different bit depths (2, 4, 6, 8) for input transformation defense

**Advanced Attacks:**
- **Targeted attacks**: Force specific misclassifications (e.g., all digits → 0)
- **Carlini-Wagner (C&W)**: Stronger optimization-based attack
- **Physical attacks**: Simulate printed adversarial examples by adding noise

**Advanced Defenses:**
- **Ensemble defenses**: Train multiple models and use majority voting
- **Randomized smoothing**: Add random noise at inference for certified robustness
- **Feature squeezing**: Combine bit depth reduction + spatial smoothing
- **Adversarial detection**: Train classifier to detect adversarial examples

**Evaluation:**
- Plot robustness curves (accuracy vs epsilon for different defenses)
- Compare computational cost (attack generation time, defense overhead)
- Test transferability across more models (logistic regression, random forest, etc.)

---

### **Additional Resources**

**Research Papers:**
- **FGSM**: "Explaining and Harnessing Adversarial Examples" (Goodfellow et al., 2015)
- **PGD**: "Towards Deep Learning Models Resistant to Adversarial Attacks" (Madry et al., 2018)
- **Transferability**: "Delving into Transferable Adversarial Examples" (Liu et al., 2016)
- **Physical Attacks**: "Robust Physical-World Attacks on Deep Learning Models" (Eykholt et al., 2018)

**Tools & Libraries:**
- **CleverHans**: Adversarial attack/defense library (TensorFlow/PyTorch)
- **Foolbox**: Adversarial robustness toolkit
- **ART (Adversarial Robustness Toolbox)**: IBM's adversarial ML library
- **RobustBench**: Leaderboard for adversarial robustness

**Standards & Guidelines:**
- **NIST AI Risk Management Framework**: Adversarial attack considerations
- **MITRE ATLAS**: Adversarial Threat Landscape for AI Systems
- **OWASP ML Security**: Top 10 ML security risks

---

### **Next Module Preview**

**Chapter 4: Privacy Attacks and Model Security**

You'll learn how attackers can **extract sensitive information** from trained models:
- **Membership inference**: Determine if specific data was in training set
- **Model inversion**: Reconstruct training data from model
- **Model extraction**: Steal model functionality
- **Privacy defenses**: Differential privacy, federated learning

These attacks target **data privacy** rather than model predictions—critical for healthcare, finance, and personal data applications!

---

**Congratulations!** You've completed the Adversarial Attacks and Model Security lab. You now understand how attackers craft imperceptible perturbations to fool ML models and how defenders can build more robust systems.